In [7]:
# =========================================
# Part 1. Hyperparameter tuning
# Keep original data interface unchanged
# =========================================

import itertools
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

# -------------------
# Basic config
# -------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Original shapes:")
print("X_train   :", getattr(X_train, "shape", None), " | y_train   :", len(y_train))
print("X_val     :", getattr(X_val, "shape", None),   " | y_val     :", len(y_val))
print("X_int     :", getattr(X_int, "shape", None),   " | y_int     :", len(y_int))
print("X_int_ext :", getattr(X_int_ext, "shape", None), " | y_int_ext :", len(y_int_ext))
print("X_ext     :", getattr(X_ext, "shape", None),   " | y_ext     :", len(y_ext))

# -------------------
# Dense versions for models that need dense input
# -------------------
def _to_dense_float32(x):
    if hasattr(x, "toarray"):
        x = x.toarray()
    x = np.asarray(x)
    if x.dtype != np.float32:
        x = x.astype(np.float32, copy=False)
    return x

X_train_dense   = _to_dense_float32(X_train)
X_val_dense     = _to_dense_float32(X_val)
X_int_dense     = _to_dense_float32(X_int)
X_int_ext_dense = _to_dense_float32(X_int_ext)
X_ext_dense     = _to_dense_float32(X_ext)

print("\nDense shapes:")
print("X_train_dense   :", X_train_dense.shape)
print("X_val_dense     :", X_val_dense.shape)
print("X_int_dense     :", X_int_dense.shape)
print("X_int_ext_dense :", X_int_ext_dense.shape)
print("X_ext_dense     :", X_ext_dense.shape)

# -------------------
# Helper functions
# -------------------
def _safe_spearman(y, pred):
    if len(y) == 0:
        return np.nan
    rho = spearmanr(y, pred)[0]
    return float(rho) if np.isfinite(rho) else np.nan

def compute_metrics(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    if len(y) == 0:
        return {
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "Spearman": np.nan
        }

    return {
        "RMSE": float(np.sqrt(mean_squared_error(y, pred))),
        "MAE": float(mean_absolute_error(y, pred)),
        "R2": float(r2_score(y, pred)) if len(y) >= 2 else np.nan,
        "Spearman": _safe_spearman(y, pred)
    }

def param_product(param_grid):
    keys = list(param_grid.keys())
    vals = list(param_grid.values())
    for combo in itertools.product(*vals):
        yield dict(zip(keys, combo))

def get_model_inputs(model_name):
    dense_models = {"SVR", "DNN"}
    if model_name in dense_models:
        return X_train_dense, X_val_dense, X_int_dense, X_int_ext_dense, X_ext_dense
    return X_train, X_val, X_int, X_int_ext, X_ext

def safe_predict(model, X):
    if X.shape[0] == 0:
        return np.array([], dtype=float)
    return model.predict(X)

def build_dnn(input_dim, hidden1=64, hidden2=32, dropout=0.2, lr=1e-3):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse"
    )
    return model

# -------------------
# Tuning: sklearn-style models
# -------------------
def tune_sklearn_model(model_name, base_model, param_grid):
    Xtr, Xva, _, _, _ = get_model_inputs(model_name)

    best_model = None
    best_params = None
    best_val_metrics = None
    best_val_rmse = np.inf

    all_results = []

    for params in param_product(param_grid):
        model = clone(base_model)
        model.set_params(**params)
        model.fit(Xtr, y_train)

        pred_val = safe_predict(model, Xva)
        val_metrics = compute_metrics(y_val, pred_val)

        row = {
            "Model": model_name,
            "Params": params,
            "Val_RMSE": val_metrics["RMSE"],
            "Val_MAE": val_metrics["MAE"],
            "Val_R2": val_metrics["R2"],
            "Val_Spearman": val_metrics["Spearman"]
        }
        all_results.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_model = model
            best_params = params
            best_val_metrics = val_metrics

    return {
        "Model": best_model,
        "BestParams": best_params,
        "Val": best_val_metrics,
        "AllResults": pd.DataFrame(all_results).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)
    }

# -------------------
# Tuning: XGBoost
# -------------------
def tune_xgb(param_grid):
    Xtr, Xva, _, _, _ = get_model_inputs("XGBoost")

    best_model = None
    best_params = None
    best_val_metrics = None
    best_val_rmse = np.inf

    all_results = []

    for params in param_product(param_grid):
        model = XGBRegressor(
            n_estimators=3000,
            random_state=SEED,
            n_jobs=-1,
            objective="reg:squarederror",
            **params
        )
        model.fit(
            Xtr, y_train,
            eval_set=[(Xva, y_val)],
            verbose=False
        )

        pred_val = safe_predict(model, Xva)
        val_metrics = compute_metrics(y_val, pred_val)

        row = {
            "Model": "XGBoost",
            "Params": params,
            "Val_RMSE": val_metrics["RMSE"],
            "Val_MAE": val_metrics["MAE"],
            "Val_R2": val_metrics["R2"],
            "Val_Spearman": val_metrics["Spearman"]
        }
        all_results.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_model = model
            best_params = params
            best_val_metrics = val_metrics

    return {
        "Model": best_model,
        "BestParams": best_params,
        "Val": best_val_metrics,
        "AllResults": pd.DataFrame(all_results).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)
    }

# -------------------
# Tuning: LightGBM
# -------------------
def tune_lgbm(param_grid):
    Xtr, Xva, _, _, _ = get_model_inputs("LightGBM")

    best_model = None
    best_params = None
    best_val_metrics = None
    best_val_rmse = np.inf

    all_results = []

    for params in param_product(param_grid):
        model = LGBMRegressor(
            n_estimators=3000,
            random_state=SEED,
            n_jobs=-1,
            **params
        )

        model.fit(
            Xtr, y_train,
            eval_set=[(Xva, y_val)],
            callbacks=[lgb_early_stopping(100, verbose=False)]
        )

        pred_val = safe_predict(model, Xva)
        val_metrics = compute_metrics(y_val, pred_val)

        row = {
            "Model": "LightGBM",
            "Params": params,
            "Val_RMSE": val_metrics["RMSE"],
            "Val_MAE": val_metrics["MAE"],
            "Val_R2": val_metrics["R2"],
            "Val_Spearman": val_metrics["Spearman"]
        }
        all_results.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_model = model
            best_params = params
            best_val_metrics = val_metrics

    return {
        "Model": best_model,
        "BestParams": best_params,
        "Val": best_val_metrics,
        "AllResults": pd.DataFrame(all_results).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)
    }

# -------------------
# Tuning: CatBoost
# -------------------
def tune_catboost(param_grid):
    Xtr, Xva, _, _, _ = get_model_inputs("CatBoost")

    best_model = None
    best_params = None
    best_val_metrics = None
    best_val_rmse = np.inf

    all_results = []

    for params in param_product(param_grid):
        model = CatBoostRegressor(
            iterations=3000,
            loss_function="RMSE",
            random_seed=SEED,
            verbose=False,
            **params
        )

        model.fit(
            Xtr, y_train,
            eval_set=(Xva, y_val),
            use_best_model=True,
            verbose=False
        )

        pred_val = safe_predict(model, Xva)
        val_metrics = compute_metrics(y_val, pred_val)

        row = {
            "Model": "CatBoost",
            "Params": params,
            "Val_RMSE": val_metrics["RMSE"],
            "Val_MAE": val_metrics["MAE"],
            "Val_R2": val_metrics["R2"],
            "Val_Spearman": val_metrics["Spearman"]
        }
        all_results.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_model = model
            best_params = params
            best_val_metrics = val_metrics

    return {
        "Model": best_model,
        "BestParams": best_params,
        "Val": best_val_metrics,
        "AllResults": pd.DataFrame(all_results).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)
    }

# -------------------
# Tuning: DNN
# -------------------
def tune_dnn(candidate_list):
    Xtr, Xva, _, _, _ = get_model_inputs("DNN")

    best_model = None
    best_params = None
    best_val_metrics = None
    best_val_rmse = np.inf
    best_history = None

    all_results = []

    for params in candidate_list:
        tf.keras.backend.clear_session()
        np.random.seed(SEED)
        tf.random.set_seed(SEED)

        model = build_dnn(
            input_dim=Xtr.shape[1],
            hidden1=params["hidden1"],
            hidden2=params["hidden2"],
            dropout=params["dropout"],
            lr=params["lr"]
        )

        es = EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        )

        history = model.fit(
            Xtr, y_train,
            validation_data=(Xva, y_val),
            epochs=300,
            batch_size=params["batch_size"],
            verbose=0,
            callbacks=[es]
        )

        pred_val = model.predict(Xva, verbose=0).ravel()
        val_metrics = compute_metrics(y_val, pred_val)

        row = {
            "Model": "DNN",
            "Params": params,
            "Val_RMSE": val_metrics["RMSE"],
            "Val_MAE": val_metrics["MAE"],
            "Val_R2": val_metrics["R2"],
            "Val_Spearman": val_metrics["Spearman"]
        }
        all_results.append(row)

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_model = model
            best_params = params
            best_val_metrics = val_metrics
            best_history = history.history

    return {
        "Model": best_model,
        "BestParams": best_params,
        "Val": best_val_metrics,
        "History": best_history,
        "AllResults": pd.DataFrame(all_results).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)
    }

# -------------------
# Search space
# -------------------
search_space = {
    "Linear Regression": (
        "sklearn",
        LinearRegression(),
        {"fit_intercept": [True, False]}
    ),
    "Ridge": (
        "sklearn",
        Ridge(),
        {"alpha": [0.01, 0.1, 1, 10, 100]}
    ),
    "Lasso": (
        "sklearn",
        Lasso(max_iter=20000, random_state=SEED),
        {"alpha": [0.0005, 0.001, 0.01, 0.1, 1]}
    ),
    "Elastic Net": (
        "sklearn",
        ElasticNet(max_iter=20000, random_state=SEED),
        {
            "alpha": [0.0005, 0.001, 0.01, 0.1, 1],
            "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
        }
    ),
    "Decision Tree": (
        "sklearn",
        DecisionTreeRegressor(random_state=SEED),
        {
            "max_depth": [2, 3, 4, 5, 6, None],
            "min_samples_leaf": [1, 2, 5, 10],
            "min_samples_split": [2, 5, 10, 20]
        }
    ),
    "SVR": (
        "sklearn",
        SVR(kernel="rbf"),
        {
            "C": [0.1, 1, 10, 30, 100],
            "gamma": ["scale", 0.001, 0.01, 0.1],
            "epsilon": [0.01, 0.1, 0.5]
        }
    ),
    "XGBoost": (
        "xgb",
        None,
        {
            "learning_rate": [0.01, 0.03, 0.05],
            "max_depth": [3, 4, 5, 6],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
            "min_child_weight": [1, 3, 5]
        }
    ),
    "LightGBM": (
        "lgbm",
        None,
        {
            "learning_rate": [0.01, 0.03, 0.05],
            "num_leaves": [15, 31, 63],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
            "min_child_samples": [5, 10, 20]
        }
    ),
    "CatBoost": (
        "cat",
        None,
        {
            "learning_rate": [0.01, 0.03, 0.05],
            "depth": [4, 5, 6, 8],
            "l2_leaf_reg": [1, 3, 5, 7]
        }
    ),
    "DNN": (
        "dnn",
        None,
        [
            {"hidden1": 64, "hidden2": 32, "dropout": 0.2, "lr": 1e-3, "batch_size": 32},
            {"hidden1": 128, "hidden2": 64, "dropout": 0.3, "lr": 1e-3, "batch_size": 32},
            {"hidden1": 64, "hidden2": 32, "dropout": 0.1, "lr": 5e-4, "batch_size": 64},
        ]
    )
}

# -------------------
# Run tuning
# -------------------
tuning_results = {}

for model_name, (model_type, base_model, grid) in search_space.items():
    print(f"\n===== Tuning {model_name} =====")

    if model_type == "sklearn":
        res = tune_sklearn_model(model_name, base_model, grid)
    elif model_type == "xgb":
        res = tune_xgb(grid)
    elif model_type == "lgbm":
        res = tune_lgbm(grid)
    elif model_type == "cat":
        res = tune_catboost(grid)
    elif model_type == "dnn":
        res = tune_dnn(grid)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    tuning_results[model_name] = res
    print("Best params:", res["BestParams"])
    print("Best Val   :", {k: round(v, 4) if np.isfinite(v) else np.nan for k, v in res["Val"].items()})

# -------------------
# Tuning summary table
# -------------------
tuning_summary = pd.DataFrame([
    {
        "Model": name,
        "BestParams": str(res["BestParams"]),
        "Val_RMSE": res["Val"]["RMSE"],
        "Val_MAE": res["Val"]["MAE"],
        "Val_R2": res["Val"]["R2"],
        "Val_Spearman": res["Val"]["Spearman"]
    }
    for name, res in tuning_results.items()
]).sort_values("Val_RMSE", ascending=True).reset_index(drop=True)

print("\n===== Tuning summary =====")
print(tuning_summary)



Original shapes:
X_train   : (5412, 423)  | y_train   : 5412
X_val     : (676, 423)  | y_val     : 676
X_int     : (677, 423)  | y_int     : 677
X_int_ext : (243, 423)  | y_int_ext : 243
X_ext     : (517, 423)  | y_ext     : 517

Dense shapes:
X_train_dense   : (5412, 423)
X_val_dense     : (676, 423)
X_int_dense     : (677, 423)
X_int_ext_dense : (243, 423)
X_ext_dense     : (517, 423)

===== Tuning Linear Regression =====
Best params: {'fit_intercept': True}
Best Val   : {'RMSE': 8.5822, 'MAE': 6.2985, 'R2': 0.8036, 'Spearman': 0.9145}

===== Tuning Ridge =====
Best params: {'alpha': 10}
Best Val   : {'RMSE': 8.5389, 'MAE': 6.315, 'R2': 0.8056, 'Spearman': 0.9145}

===== Tuning Lasso =====
Best params: {'alpha': 0.01}
Best Val   : {'RMSE': 8.5163, 'MAE': 6.2873, 'R2': 0.8066, 'Spearman': 0.9169}

===== Tuning Elastic Net =====
Best params: {'alpha': 0.01, 'l1_ratio': 0.5}
Best Val   : {'RMSE': 8.4816, 'MAE': 6.3466, 'R2': 0.8082, 'Spearman': 0.9153}

===== Tuning Decision Tree =====
